In [8]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

class SP500HighsStrategy:
    def __init__(self, initial_capital=100000, stop_loss_multiplier=2):
        self.initial_capital = initial_capital
        self.current_capital = initial_capital
        self.cash_balance = initial_capital  # 現金餘額
        self.stop_loss_multiplier = stop_loss_multiplier
        self.positions = {}
        self.portfolio_history = []
        self.sp500_symbols = self.get_sp500_symbols()
        
    def get_sp500_symbols(self):
        """獲取S&P 500股票代碼列表"""
        # 這裡使用一個簡化的S&P 500股票列表
        # 實際應用中可以從網站爬取最新的完整列表
        symbols = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies')[0]['Symbol'].tolist()[:500]
        return symbols
    
    def get_stock_data(self, symbol, period='10y'):
        """獲取股票歷史數據"""
        try:
            stock = yf.Ticker(symbol)
            data = stock.history(period=period)
            if data.empty:
                print(f"警告: {symbol} 沒有數據")
                return None
            return data
        except Exception as e:
            print(f"無法獲取 {symbol} 的數據: {e}")
            return None
    
    def calculate_52week_high(self, data):
        """計算52週最高價"""
        if len(data) < 252:  # 如果數據不足52週
            return data['High'].max()
        rolling_max = data['High'].rolling(window=252, min_periods=1).max()
        return rolling_max.iloc[-1] if hasattr(rolling_max, 'iloc') else rolling_max
    
    def calculate_volatility(self, data, window=20):
        """計算股票標準差（波動率）"""
        returns = data['Close'].pct_change().dropna()
        if len(returns) < window:
            return 0.2  # 預設20%
        volatility = returns.rolling(window=window, min_periods=1).std() * np.sqrt(252)  # 年化標準差
        return volatility.iloc[-1] if not volatility.empty and hasattr(volatility, 'iloc') else 0.2
    
    def is_52week_high_breakout(self, data):
        """檢查是否突破52週新高"""
        if len(data) < 2:
            return False
        
        current_price = data['Close'].iloc[-1]
        # 獲取前一天之前的數據來計算52週最高價
        previous_data = data.iloc[:-1]
        if len(previous_data) == 0:
            return False
            
        previous_52w_high = self.calculate_52week_high(previous_data)
        
        return current_price > previous_52w_high
    
    def screen_stocks_weekly(self, date):
        """每週篩選突破52週新高的股票"""
        breakout_stocks = []
        print(f"正在篩選股票...")
        
        for i, symbol in enumerate(self.sp500_symbols[:20]):  # 限制為前20隻股票以加快測試
            if i % 5 == 0:
                print(f"處理進度: {i+1}/{min(20, len(self.sp500_symbols))}")
                
            data = self.get_stock_data(symbol)
            
            if data is None or len(data) < 50:
                continue
            
            try:
                # 檢查是否突破52週新高
                if self.is_52week_high_breakout(data):
                    volatility = self.calculate_volatility(data)
                    current_price = data['Close'].iloc[-1]
                    
                    breakout_stocks.append({
                        'symbol': symbol,
                        'price': current_price,
                        'volatility': volatility,
                        'date': date
                    })
                    print(f"發現突破: {symbol} @ ${current_price:.2f}, 波動率: {volatility:.3f}")
            except Exception as e:
                print(f"處理 {symbol} 時發生錯誤: {e}")
                continue
        
        return breakout_stocks
    
    def calculate_position_weights(self, breakout_stocks):
        """根據標準差分配權重（標準差越低，權重越高）"""
        if not breakout_stocks:
            return {}
        
        # 計算權重（標準差的倒數，然後標準化）
        volatilities = [stock['volatility'] for stock in breakout_stocks]
        inverse_vol = [1/vol if vol > 0 else 1 for vol in volatilities]
        total_inverse_vol = sum(inverse_vol)
        
        weights = {}
        for i, stock in enumerate(breakout_stocks):
            weight = inverse_vol[i] / total_inverse_vol
            weights[stock['symbol']] = {
                'weight': weight,
                'price': stock['price'],
                'volatility': stock['volatility'],
                'stop_loss': stock['price'] * (1 - self.stop_loss_multiplier * stock['volatility'])
            }
        
        return weights
    
    def execute_trades(self, new_weights, current_date):
        """執行交易"""
        # 計算現有持仓的價值並轉為現金
        current_portfolio_value = self.calculate_portfolio_value()
        self.cash_balance = current_portfolio_value
        
        # 清空所有現有持仓
        self.positions.clear()
        
        if not new_weights:
            print(f"\n{current_date} 交易執行:")
            print("沒有符合條件的股票，資金保持現金形式")
            print(f"現金餘額: ${self.cash_balance:,.2f}")
            return
        
        # 根據新權重建立持仓
        total_invested = 0
        for symbol, info in new_weights.items():
            position_value = self.cash_balance * info['weight']
            shares = int(position_value / info['price'])
            
            if shares > 0:
                actual_cost = shares * info['price']
                self.positions[symbol] = {
                    'shares': shares,
                    'entry_price': info['price'],
                    'stop_loss': info['stop_loss'],
                    'volatility': info['volatility'],
                    'entry_date': current_date
                }
                total_invested += actual_cost
        
        # 更新現金餘額
        self.cash_balance -= total_invested
        
        print(f"\n{current_date} 交易執行:")
        print(f"總持仓數量: {len(self.positions)}")
        print(f"已投資金額: ${total_invested:,.2f}")
        print(f"剩餘現金: ${self.cash_balance:,.2f}")
        for symbol, pos in self.positions.items():
            print(f"{symbol}: {pos['shares']}股 @ ${pos['entry_price']:.2f}, 止損: ${pos['stop_loss']:.2f}")
    
    def check_stop_losses(self, current_date):
        """檢查止損條件"""
        positions_to_close = []
        cash_from_sales = 0
        
        for symbol in self.positions:
            data = self.get_stock_data(symbol, period='5d')
            if data is None or len(data) == 0:
                continue
                
            current_price = data['Close'].iloc[-1]
            position = self.positions[symbol]
            
            if current_price <= position['stop_loss']:
                positions_to_close.append(symbol)
                # 計算賣出後的現金
                sale_value = position['shares'] * current_price
                cash_from_sales += sale_value
                print(f"{current_date}: {symbol} 觸發止損 - 當前價格: ${current_price:.2f}, 止損價格: ${position['stop_loss']:.2f}, 賣出價值: ${sale_value:,.2f}")
        
        # 關閉觸發止損的持仓並更新現金餘額
        for symbol in positions_to_close:
            del self.positions[symbol]
        
        self.cash_balance += cash_from_sales
    
    def calculate_portfolio_value(self):
        """計算投資組合總價值（持股價值 + 現金）"""
        total_stock_value = 0
        
        for symbol, position in self.positions.items():
            data = self.get_stock_data(symbol, period='5d')
            if data is None or len(data) == 0:
                # 如果無法獲取數據，使用買入價格
                current_price = position['entry_price']
            else:
                current_price = data['Close'].iloc[-1]
            
            position_value = position['shares'] * current_price
            total_stock_value += position_value
        
        # 總投資組合價值 = 持股價值 + 現金餘額
        return total_stock_value + self.cash_balance
    
    def run_backtest(self, start_date='2014-01-01', end_date='2024-01-01'):
        """運行回測"""
        print("開始運行S&P 500 52週新高策略...")
        print(f"初始資本: ${self.initial_capital:,.2f}")
        print(f"止損倍數: {self.stop_loss_multiplier}")
        
        # 獲取週五日期列表
        current_date = datetime.strptime(start_date, '%Y-%m-%d')
        end_date = datetime.strptime(end_date, '%Y-%m-%d')
        
        fridays = []
        while current_date <= end_date:
            if current_date.weekday() == 4:  # 週五
                fridays.append(current_date.strftime('%Y-%m-%d'))
            current_date += timedelta(days=1)
        
        # 每週執行策略
        for friday in fridays[:10]:  # 限制為前10週以加快演示
            print(f"\n{'='*50}")
            print(f"處理週期: {friday}")
            
            # 篩選突破股票
            breakout_stocks = self.screen_stocks_weekly(friday)
            print(f"發現 {len(breakout_stocks)} 隻突破52週新高的股票")
            
            # 計算權重並執行交易
            if breakout_stocks:
                new_weights = self.calculate_position_weights(breakout_stocks)
            else:
                new_weights = {}
            
            self.execute_trades(new_weights, friday)
            
            # 檢查止損
            self.check_stop_losses(friday)
            
            # 計算當前投資組合價值
            portfolio_value = self.calculate_portfolio_value()
            stock_value = portfolio_value - self.cash_balance
            
            self.portfolio_history.append({
                'date': friday,
                'portfolio_value': portfolio_value,
                'stock_value': stock_value,
                'cash_balance': self.cash_balance,
                'positions_count': len(self.positions)
            })
            
            print(f"投資組合總價值: ${portfolio_value:,.2f}")
            print(f"  - 持股價值: ${stock_value:,.2f}")
            print(f"  - 現金餘額: ${self.cash_balance:,.2f}")
            print(f"總收益率: {((portfolio_value/self.initial_capital)-1)*100:.2f}%")
    
    def generate_report(self):
        """生成策略報告"""
        if not self.portfolio_history:
            print("沒有交易歷史記錄")
            return
        
        print(f"\n{'='*60}")
        print("策略績效報告")
        print(f"{'='*60}")
        
        final_value = self.portfolio_history[-1]['portfolio_value']
        total_return = (final_value / self.initial_capital - 1) * 100
        
        print(f"初始資本: ${self.initial_capital:,.2f}")
        print(f"最終價值: ${final_value:,.2f}")
        print(f"總收益率: {total_return:.2f}%")
        
        # 顯示投資組合歷史
        print(f"\n投資組合歷史:")
        print("-" * 60)
        print(f"{'日期':<12} {'總價值':<12} {'持股價值':<12} {'現金':<12} {'持仓數'}")
        print("-" * 60)
        for record in self.portfolio_history:
            print(f"{record['date']:<12} ${record['portfolio_value']:<11,.0f} "
                  f"${record['stock_value']:<11,.0f} ${record['cash_balance']:<11,.0f} "
                  f"{record['positions_count']}")

# 使用範例
if __name__ == "__main__":
    # 創建策略實例
    strategy = SP500HighsStrategy(initial_capital=100000, stop_loss_multiplier=2)
    
    # 運行回測
    strategy.run_backtest(start_date='2014-01-01', end_date='2023-12-31')
    
    # 生成報告
    strategy.generate_report()
    
    print(f"\n當前持仓:")
    for symbol, position in strategy.positions.items():
        print(f"{symbol}: {position['shares']}股 @ ${position['entry_price']:.2f}")

開始運行S&P 500 52週新高策略...
初始資本: $100,000.00
止損倍數: 2

處理週期: 2014-01-03
正在篩選股票...
處理進度: 1/20
處理進度: 6/20
處理進度: 11/20
處理進度: 16/20
發現 0 隻突破52週新高的股票

2014-01-03 交易執行:
沒有符合條件的股票，資金保持現金形式
現金餘額: $100,000.00
投資組合總價值: $100,000.00
  - 持股價值: $0.00
  - 現金餘額: $100,000.00
總收益率: 0.00%

處理週期: 2014-01-10
正在篩選股票...
處理進度: 1/20
處理進度: 6/20
處理進度: 11/20
處理進度: 16/20
發現 0 隻突破52週新高的股票

2014-01-10 交易執行:
沒有符合條件的股票，資金保持現金形式
現金餘額: $100,000.00
投資組合總價值: $100,000.00
  - 持股價值: $0.00
  - 現金餘額: $100,000.00
總收益率: 0.00%

處理週期: 2014-01-17
正在篩選股票...
處理進度: 1/20
處理進度: 6/20
處理進度: 11/20
處理進度: 16/20
發現 0 隻突破52週新高的股票

2014-01-17 交易執行:
沒有符合條件的股票，資金保持現金形式
現金餘額: $100,000.00
投資組合總價值: $100,000.00
  - 持股價值: $0.00
  - 現金餘額: $100,000.00
總收益率: 0.00%

處理週期: 2014-01-24
正在篩選股票...
處理進度: 1/20
處理進度: 6/20
處理進度: 11/20
處理進度: 16/20
發現 0 隻突破52週新高的股票

2014-01-24 交易執行:
沒有符合條件的股票，資金保持現金形式
現金餘額: $100,000.00
投資組合總價值: $100,000.00
  - 持股價值: $0.00
  - 現金餘額: $100,000.00
總收益率: 0.00%

處理週期: 2014-01-31
正在篩選股票...
處理進度: 1/20
處理進度: 6/20
處理進度: 11/20
處理進度: 16/20
發現 0 隻突破5